# Vocabulary Mapping Benchmarking

This notebook demonstrates how to benchmark SNOMED CT vocabulary mapping methods (ICD-10, LOINC, etc.).

## Overview

Vocabulary mapping evaluates how well a method can convert SNOMED CT concepts to codes in other terminologies.

### Key Metrics:
- **Precision@K**: Fraction of top-K mapped codes that are correct
- **Recall@K**: Fraction of expected mappings found in top-K
- **Coverage Rate**: Fraction of all expected mappings successfully found
- **MRR (Mean Reciprocal Rank)**: Average rank of first correct mapping
- **F1@K**: Harmonic mean of precision and recall at K

### Data Format:
```python
{
    'snomed_cui': str,         # SNOMED CT concept ID
    'target_codes': List[str]  # Expected mapped codes
}
```

In [ ]:
# Import required modules
from snomed_methods.benchmarking.vocabulary import (
    evaluate_mapper,
    generate_mapping_dataset,
)

## Generate Synthetic Mapping Dataset

In [ ]:
# Generate vocabulary mapping dataset
dataset = generate_mapping_dataset(num_samples=50)

print(f"Dataset size: {len(dataset)}")
print("\nFirst sample:")
sample = dataset[0]
print(f"  SNOMED CUI: {sample['snomed_cui']}")
print(f"  Target vocabulary: {sample.get('vocab_type', 'N/A')}")
print(f"  Expected codes ({len(sample['target_codes'])}):")
for code in sample["target_codes"]:
    print(f"    - {code}")

## Create Mock Mapper Function

For demonstration, we create a simple mapping function.

In [ ]:
# Example: Simple mock mapper based on hash patterns
def simple_mapper(snomed_cui: str) -> list:
    """Mock mapper returning synthetic ICD-10 and LOINC codes."""
    seed_hash = hash(snomed_cui)

    # Generate mappings deterministically
    icd_codes = [
        f"ICD_E{abs(seed_hash + 1) % 99:02d}.{abs(seed_hash + 2) % 9}" for _ in range(3)
    ]
    loinc_codes = [f"LOINC_{718 + abs(seed_hash) % 100}" for _ in range(2)]

    # Combine and return top results
    mappings = icd_codes + loinc_codes
    return mappings[:5]


# Test the mock mapper
test_cui = dataset[0]["snomed_cui"]
result = simple_mapper(test_cui)
print(f"SNOMED CUI: {test_cui}")
print(f"Mapped codes ({len(result)}):")
for code in result:
    print(f"  - {code}")

## Evaluate Vocabulary Mapper

In [ ]:
# Evaluate the mapper
results = evaluate_mapper(
    mapper_func=simple_mapper,
    dataset=dataset,
    k_values=[1, 3, 5],
)

print("\n=== Vocabulary Mapping Benchmarking Results ===")

# Collect all metrics for display
metrics_to_show = [
    ("coverage_rate", "Coverage Rate"),
    ("mrr", "MRR"),
]

for key, label in metrics_to_show:
    if key in results:
        print(f"{label}: {results[key]:.4f}")

print("\nPrecision and Recall at different K values:")
print(f"{'K':<5} {'Precision':<12} {'Recall':<12} {'F1':<10}")
print("-" * 45)

for k in [1, 3, 5]:
    p = results.get(f"precision@{k}", 0)
    r = results.get(f"recall@{k}", 0)
    f1 = results.get(f"f1@{k}", 0)
    print(f"{k:<5} {p:<12.4f} {r:<12.4f} {f1:<10.4f}")

print(f"\nTotal samples evaluated: {results['num_samples']}")

## Working with Real SNOMED Data

When actual SNOMED CT mapping files are available:

In [ ]:
# Example: Using real UMLSCIMapper (requires RF2 data)
# from snomed_methods import UMLSCIMapper

# mapper = UMLSCIMapper(
#     uk_path="/path/to/uk_sct2cl_42.2.0",
# )

# def real_mapper(snomed_cui):
#     result = mapper.map_to_umls(snomed_cui)
#     # Extract ICD-10 or LOINC mappings if available
#     codes = []
#     for m in result:
#         if 'umls_cui' in m:
#             codes.append(f"UMLS_{m['umls_cui']}")
#     return codes[:5]

# results_real = evaluate_mapper(real_mapper, dataset[:10])
# print(results_real)

## Load Pre-generated Datasets

In [ ]:
from snomed_methods.benchmarking.vocabulary import load_mapping_datasets

# Load all pre-generated datasets
datasets = load_mapping_datasets()

for name, data in datasets.items():
    print(f"{name}: {len(data)} samples")

## Coverage Analysis

Analyze how many expected mappings are successfully found.

In [ ]:
# Detailed coverage analysis per sample
print("\nSample-level coverage analysis:")

for i, sample in enumerate(dataset[:5]):
    snomed_cui = sample["snomed_cui"]
    expected = set(sample["target_codes"])
    predicted = simple_mapper(snomed_cui)

    pred_set = set(predicted)
    intersection = len(expected & pred_set)
    union = len(expected | pred_set)
    jaccard = intersection / union if union else 0

    print(f"\nSample {i+1}:")
    print(f"  Expected: {len(expected)} codes")
    print(
        f"  Found: {intersection}/{len(expected)} ({intersection/len(expected)*100:.1f}%)"
    )